# Pedal-aware eval — full unattended runRuns the whole chain end to end so it can be left overnight:1. **Synthesise** the 50 eval WAVs **with the sustain pedal** (CC64 from MAESTRO)2. **Run the pipeline** on all 503. **Score transcription** and save a new baseline4. **Score the pedal** — for the first time ever### Why this mattersThe eval audio was previously rendered with CC64 stripped. MAESTRO performanceshave the damper **down 63.8% of the time**, so our test set has beensystematically easier than real playing in exactly the dimension that blursharmonics and overlaps decay.**Expect COnP to fall.** That fall is correct — the old numbers were measuredagainst audio no pianist would produce. Old baselines are not comparable acrossthis change, which is why step 3 saves a fresh one rather than diffing.

In [ ]:
# ── setup: everything needed is already in backend/venv, this only verifies itimport sys, os, json, time, subprocess, pathlibROOT = pathlib.Path.cwd()while not (ROOT / 'backend').exists() and ROOT != ROOT.parent:    ROOT = ROOT.parentos.chdir(ROOT)PY_EXE = str(ROOT / 'backend' / 'venv' / 'Scripts' / 'python.exe')if not os.path.exists(PY_EXE):    PY_EXE = sys.executablesys.path.insert(0, str(ROOT / 'backend'))print('repo :', ROOT)print('python:', PY_EXE)for mod in ('numpy','librosa','soundfile','pretty_midi','mir_eval','torch'):    try:        __import__(mod); print(f'  {mod:<12} ok')    except Exception as e:        print(f'  {mod:<12} MISSING -> {e}')

In [ ]:
# ── helper: stream a subprocess so the notebook shows progress livedef run(args, tag):    print(f'=== {tag} ===', flush=True)    t0 = time.time()    p = subprocess.Popen([PY_EXE, '-u', *args], cwd=str(ROOT),                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT,                         text=True, encoding='utf-8', errors='replace',                         env={**os.environ, 'PYTHONIOENCODING': 'utf-8'})    for line in p.stdout:        line = line.rstrip()        if line:            print(line, flush=True)    p.wait()    print(f'--- {tag}: exit {p.returncode} in {(time.time()-t0)/60:.1f} min', flush=True)    return p.returncode# -u matters: without unbuffered output a piped Python prints nothing until# it exits, which is how an earlier run of this looked frozen for an hour.

## 1 · Synthesise with pedal  *(~45–60 min)*

In [ ]:
run(['backend/eval/synthesize_audio.py', '--force'], 'synthesise with pedal')

## 2 · Run the pipeline on all 50  *(~40 min)*

In [ ]:
run(['backend/eval/run_pipeline.py', '--all', '--force'], 'pipeline')

## 3 · Transcription scores + fresh baseline

In [ ]:
run(['backend/eval/compare.py', '--all', '--save-baseline'], 'compare + baseline')

## 4 · Pedal accuracy — the new measurementFrame-level at 10 ms, because pedal is a binary *state* over time rather than aset of events: two pianists can produce very different numbers of presses andthe damper still be in the same place, which event-matching would score as atotal mismatch.Read `frame F1` **next to** `f1_always_down`. With truth ~64% down, a detectorthat simply held the pedal forever scores about 0.78 — so anything near that isworthless regardless of how good it looks.

In [ ]:
import pathlib, jsonsys.path.insert(0, str(ROOT / 'backend' / 'eval'))from eval.pedal_metrics import pedal_scores, aggregate, reportfrom eval.synthesize_audio import _midi_to_pedalman = json.load(open(ROOT / 'backend/eval/data/manifest.json'))pieces = man['pieces'] if isinstance(man, dict) and 'pieces' in man else manrows, missing = [], 0for p in pieces:    run_dir = ROOT / 'backend/eval/runs' / p['id']    tf = run_dir / 'transcription.json'    if not tf.exists():        missing += 1; continue    data = json.load(open(tf))    pred = data.get('pedal') or []    truth = _midi_to_pedal(pathlib.Path(p['midi_path']))    dur = float(p['duration_s'])    if not truth:        continue    r = pedal_scores(pred, truth, dur)    r['id'] = p['id']    rows.append(r)agg = aggregate(rows)print(report(agg))print(f'pieces scored {len(rows)}, missing runs {missing}')worst = sorted(rows, key=lambda r: r['f1'])[:5]print('worst 5 by frame F1:')for r in worst:    print(f"  {r['f1']:.3f}  down pred {r['down_rate_pred']:.2f} / truth "          f"{r['down_rate_truth']:.2f}   {r['id'][:54]}")json.dump({'aggregate': agg, 'per_piece': rows},          open(ROOT / 'backend/eval/data/pedal_scores.json', 'w'), indent=1)print('wrote backend/eval/data/pedal_scores.json')

## 5 · What to look at in the morning- **Transcription** — COnP will be lower than the old ~0.84. Expected: the audio  now has pedal in it. The number to keep is the *new* baseline, not the delta.- **Pedal frame F1 vs `f1_always_down`** — this is the real question. Comfortably  above it means the pedal head genuinely works and the pedalling-rules idea can  be judged against a real signal. Near or below it means the head is close to  useless on this material and rules would be building on sand.- **`down_rate_pred` vs `down_rate_truth`** — if ours is much higher, we  over-hold, which is the most likely cause of the residual tone-cutting and  the phantom-crowding already measured in `offset_inflation.py`.